In [ ]:
input_data = None
input_doc = None
output_data = None
util = None
display_util = None

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib_inline

from IPython.display import Markdown

matplotlib_inline.backend_inline.set_matplotlib_formats("svg")


%matplotlib inline
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)

sys.path.append(str(Path(util).parent))
sys.path.append(str(Path(display_util).parent))

In [ ]:
from display_util import (  # noqa: E402
    rename,
    display_data_doc,
)
from util import (  # noqa: E402
    drop_col_few_distinct,
    drop_duplicate_columns,
)

# empfaenger_immunologie

The {term}`ET` foundation provides immunologic test results for potential recipients in `element_empfaenger_immunologie.csv`. Depending on the test applied, different columns are used together.

We perform the common steps outlined in [](../02_preprocessing/index.md), but besides translations no conversion and filtering are applied. This data is in the longitudinal format and has a column to differentiate between different test types. 

## Unprocessed input data

In [ ]:
germannumericcols = ["EImmDonorFrequencyETKASET", "EImmDonorFrequencyHET"]
data = pd.read_csv(input_data, sep=";", low_memory=False)
for col in germannumericcols:
    data[col] = pd.to_numeric(
        data[col].str.replace(",", ".", regex=False), errors="raise"
    )
germannumericcols = [f"`{col}`" for col in germannumericcols]
display(
    Markdown(
        f"The columns {', '.join(germannumericcols)} were saved with a `,` as the decimal seperator, this was reconsiled during reading the data."
    )
)
display_data_doc(data=data, official_doc=pd.read_csv(input_doc))

## Technical Steps

For this file the general plan for technical preprocessing was followed.

### Removal of Non-Informative Columns

First empty and duplicated columns were removed (see [](general:ecf)). Furthermore we removed the column `EImmDiagnostikZentrumET`, because it contains encoded identifying information. 

In [ ]:
data = drop_col_few_distinct(data)
data = drop_duplicate_columns(data).drop(columns="EImmDiagnostikZentrumET")

### Renaming of Columns

New names were used for the columns. (see [](general:cr)).

In [ ]:
renaming = {
    "EImmAntigenAkzeptabelET": "antigens_acceptable",
    "EImmAntigenInakzeptabelET": "antigens_unacceptable",
    "EImmAntikoerperNichtZytotoxischET": "non_cytotoxic_antibodies",
    "EImmAutoantikoerperET": "auto_antibodies",
    "EImmCrossMatchDTTET": "dtt_crossmatch",
    "EImmDiagnostikScreeningverfahrenET": "screening",
    "EImmDonorFrequencyETKASET": "donor_freq_etkas",
    "EImmDonorFrequencyHET": "donor_freq_het",
    "EImmEingabeDateET": "enter_date",
    "EImmErgebnisTypET": "result_type",
    "EImmHLAPhaenotypisierungET": "hla_phenotyping",
    "EImmIdEmpfaengerNrETET": "recipient_et_id_et",
    "EImmPRAWertET": "pra_percent",
    "EImmPRAEinheitET": "pra_unit",
    "EImmvPRAWertET": "vpra_percent",
    "EImmvPRAEinheitET": "vpra_unit",
    "EImmProbeDateET": "sampling_date",
    "EImmSpezifitaetenET": "specificities",
}

data = rename(data, renaming)
data.to_parquet(output_data)